# Music Recommendation System

Advanced interactive music recommendation using Cosine Similarity with diversity constraints.

This notebook:
1. **Sample & Rate**: Presents 5 random songs for you to listen and rate (Like/Dislike/Skip)
2. **Intelligent Filtering**: Removes disliked artists and albums from recommendations
3. **Diversity-Aware**: Generates 10 diverse recommendations balancing similarity with artist/genre variety
4. **Feature Analysis**: Compares audio characteristics of your likes vs recommendations
5. **Cohesion Metrics**: Analyzes playlist consistency and inter-track similarity
6. **Export & Share**: Saves your playlist with preview URLs and Spotify URIs

**Algorithm**: Content-based filtering using Cosine Similarity on normalized audio features (8 dimensions) + genre encoding (one-hot)

In [2]:
#import sys
#!{sys.executable} -m pip install plotly nbformat

In [3]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
import logging
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Audio, HTML, display

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Add project root to path for imports
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment and config
load_dotenv(project_root / '.env', override=True)
from soeSpotify import config

# Styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
palette = sns.color_palette("viridis", as_cmap=False)

print("Setup complete.")

Setup complete.


## 1. Data Loading and Preprocessing

In [4]:
print("Loading analytical datasets...")

# Load processed data
df_tracks = pd.read_parquet(config.ANALYTICS_TRACKS)
df_features = pd.read_parquet(config.ANALYTICS_TRACK_FEATURES)

print(f"Tracks loaded: {len(df_tracks):,}")
print(f"Features loaded: {len(df_features):,}")
print(f"\nTrack columns: {list(df_tracks.columns)[:10]}...")

Loading analytical datasets...
Tracks loaded: 94,065
Features loaded: 94,065

Track columns: ['track_id', 'track_name', 'track_popularity', 'duration_ms', 'preview_url', 'uri', 'album_id', 'album_name', 'album_type', 'release_date']...


In [5]:
# Deduplicate tracks: Keep highest popularity when track appears multiple times
print("Deduplicating tracks...")
df_tracks_original_count = len(df_tracks)
df_tracks = df_tracks.sort_values('track_popularity', ascending=False)
df_tracks = df_tracks.drop_duplicates(subset=['track_name', 'artist_name'], keep='first')
df_tracks = df_tracks[df_tracks['preview_url'].notna()]

print(f"Original tracks: {df_tracks_original_count:,}")
print(f"After dedup: {len(df_tracks):,}")
print(f"Tracks with preview: {df_tracks['preview_url'].notna().sum():,}")

# Define audio features for similarity calculation
audio_features = [
    'acousticness',
    'danceability',
    'energy',
    'instrumentalness',
    'liveness',
    'loudness',
    'tempo',
    'valence'
]

# One-hot encode genre
df_encoded = pd.get_dummies(
    df_tracks,
    columns=['artist_primary_genre_broad'],
    prefix='genre'
)
genre_features = [c for c in df_encoded.columns if c.startswith('genre_')]

# Combine all features
all_features = audio_features + genre_features

print(f"\nTotal features for similarity: {len(all_features)}")
print(f"  - Audio features: {len(audio_features)}")
print(f"  - Genre features: {len(genre_features)}")

Deduplicating tracks...
Original tracks: 94,065
After dedup: 89,749
Tracks with preview: 89,749

Total features for similarity: 22
  - Audio features: 8
  - Genre features: 14


In [6]:
# Prepare feature matrix
df_features_matrix = df_encoded[all_features].copy()
df_features_matrix.index = df_encoded.index

# Handle missing values
df_features_matrix = df_features_matrix.fillna(df_features_matrix.mean())

# Standardize audio features
scaler = StandardScaler()
df_features_scaled = df_features_matrix.copy()
df_features_scaled[audio_features] = scaler.fit_transform(
    df_features_matrix[audio_features]
)

# Apply genre weight (0.5 as per model tests)
genre_weight = 0.5
df_features_scaled[genre_features] = df_features_scaled[genre_features] * genre_weight

print(f"Feature matrix shape: {df_features_scaled.shape}")
print(f"Missing values: {df_features_scaled.isnull().sum().sum()}")

Feature matrix shape: (89749, 22)
Missing values: 0


## 2. Sample 5 Random Songs for Rating

In [7]:
# Sample 5 random songs with valid preview URLs
np.random.seed(42)
tracks_with_preview = df_tracks[df_tracks['preview_url'].notna()].copy()

sample_tracks = tracks_with_preview.sample(n=10, random_state=None)
sample_track_ids = sample_tracks.index.tolist()

print("5 Random Tracks for Rating:")
print("=" * 80)
for idx, (track_id, track) in enumerate(sample_tracks.iterrows(), 1):
    print(f"\n{idx}. {track['track_name']}")
    print(f"   Artist: {track['artist_name']}")
    print(f"   Album: {track['album_name']}")
    print(f"   Genre: {track['artist_primary_genre_broad']}")
    print(f"   Popularity: {track['track_popularity']}/100")
    print(f"   Duration: {track['duration_ms'] // 60000}:{(track['duration_ms'] % 60000) // 1000:02d}")

5 Random Tracks for Rating:

1. Pretty Hands
   Artist: Jeffrey Foucault
   Album: Blood Brothers
   Genre: world
   Popularity: 48/100
   Duration: 2:20

2. I Understand I Do Not Understand Do You Understand?
   Artist: Travelanguage
   Album: English To Swedish
   Genre: 
   Popularity: 0/100
   Duration: 0:09

3. Big Jet Plane
   Artist: Alok
   Album: Big Jet Plane
   Genre: electronic
   Popularity: 71/100
   Duration: 3:02

4. Puller Opp
   Artist: G.Flow
   Album: Puller Opp
   Genre: 
   Popularity: 27/100
   Duration: 3:36

5. Outside
   Artist: ËMMË
   Album: Outside
   Genre: 
   Popularity: 52/100
   Duration: 3:12

6. 2012
   Artist: Whoja Vu
   Album: 2012
   Genre: 
   Popularity: 13/100
   Duration: 3:34

7. Sugar
   Artist: Rap Bang Club
   Album: Sugar
   Genre: hip hop
   Popularity: 31/100
   Duration: 3:16

8. Te Quiero un Poco
   Artist: Carlos Sadness
   Album: Diferentes Tipos de Luz
   Genre: indie
   Popularity: 62/100
   Duration: 3:26

9. Caught In The Rain


## 3. Interactive Rating Interface

In [8]:
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Button, ToggleButton, Output

# Store user ratings
user_ratings = {}
current_track_idx = 0

# Create widgets
track_display = widgets.HTML()
preview_player = widgets.HTML()
like_button = widgets.Button(
    description='Like',
    button_style='success',
    icon='thumbs-up'
)
dislike_button = widgets.Button(
    description='Dislike',
    button_style='danger',
    icon='thumbs-down'
)
skip_button = widgets.Button(
    description='Skip',
    button_style='info'
)

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=5,
    step=1,
    description='Progress:',
    bar_style='info'
)

output_area = Output()

def update_track_display():
    """Update the display with current track info and preview player."""
    if current_track_idx >= len(sample_track_ids):
        track_display.value = "<h3>All songs rated! Click 'Generate Playlist' to see recommendations.</h3>"
        preview_player.value = ""
        return
    
    track_id = sample_track_ids[current_track_idx]
    track = df_tracks.loc[track_id]
    
    # Track info HTML
    html = f"""
    <div style="border: 2px solid #007bff; padding: 15px; border-radius: 10px;">
        <h3>{track['track_name']}</h3>
        <p><strong>Artist:</strong> {track['artist_name']}</p>
        <p><strong>Album:</strong> {track['album_name']}</p>
        <p><strong>Genre:</strong> {track['artist_primary_genre_broad']}</p>
        <p><strong>Popularity:</strong> {track['track_popularity']}/100</p>
        <p><strong>Duration:</strong> {track['duration_ms'] // 60000}:{(track['duration_ms'] % 60000) // 1000:02d}</p>
    </div>
    """
    track_display.value = html
    
    # Preview player HTML
    if pd.notna(track['preview_url']):
        preview_html = f"""
        <div style="margin-top: 15px;">
            <p><strong>Preview:</strong></p>
            <audio controls style="width: 100%;">
                <source src="{track['preview_url']}" type="audio/mpeg">
                Your browser does not support the audio element.
            </audio>
        </div>
        """
        preview_player.value = preview_html
    else:
        preview_player.value = "<p>No preview available for this track.</p>"

def on_like_clicked(b):
    """Handle like button click."""
    global current_track_idx
    track_id = sample_track_ids[current_track_idx]
    user_ratings[track_id] = 1  # 1 for like
    current_track_idx += 1
    progress.value = current_track_idx
    update_track_display()

def on_dislike_clicked(b):
    """Handle dislike button click."""
    global current_track_idx
    track_id = sample_track_ids[current_track_idx]
    user_ratings[track_id] = -1  # -1 for dislike
    current_track_idx += 1
    progress.value = current_track_idx
    update_track_display()

def on_skip_clicked(b):
    """Handle skip button click."""
    global current_track_idx
    track_id = sample_track_ids[current_track_idx]
    user_ratings[track_id] = 0  # 0 for skip/neutral
    current_track_idx += 1
    progress.value = current_track_idx
    update_track_display()

# Attach event handlers
like_button.on_click(on_like_clicked)
dislike_button.on_click(on_dislike_clicked)
skip_button.on_click(on_skip_clicked)

# Initial display
update_track_display()

# Layout
button_box = HBox(
    [like_button, dislike_button, skip_button],
    layout=widgets.Layout(gap='10px')
)
interface_box = VBox(
    [progress, track_display, preview_player, button_box],
    layout=widgets.Layout(gap='10px', padding='10px')
)

display(interface_box)

## 4. Generate Personalized Playlist with Advanced Engine

In [13]:
def get_recommendations_advanced(
    user_ratings,
    df_tracks,
    df_features_scaled,
    n_recommendations=10,
    diversity_weight=0.3,
    min_artists=None,
    min_genres=None
):
    """
    Advanced recommendation engine with diversity constraints.
    
    Balances similarity scores with diversity to avoid repetitive recommendations.
    
    Parameters:
    -----------
    user_ratings : dict
        Dictionary with track_id as key and rating (1/-1/0) as value
    df_tracks : pd.DataFrame
        Track metadata
    df_features_scaled : pd.DataFrame
        Scaled feature matrix
    n_recommendations : int
        Number of recommendations to return
    diversity_weight : float
        Weight for diversity penalty (0-1). Higher = more diverse
    min_artists : int
        Minimum unique artists in playlist (None for no constraint)
    min_genres : int
        Minimum unique genres in playlist (None for no constraint)
    
    Returns:
    --------
    pd.DataFrame : Top recommendations with diversity boost
    """
    # Separate likes and dislikes
    liked_ids = [tid for tid, rating in user_ratings.items() if rating == 1]
    disliked_ids = [tid for tid, rating in user_ratings.items() if rating == -1]
    
    if len(liked_ids) == 0:
        print("No liked tracks. Cannot generate recommendations.")
        return pd.DataFrame()
    
    # Compute mean profile of liked tracks
    liked_profiles = df_features_scaled.loc[liked_ids]
    user_profile = liked_profiles.mean(axis=0).values.reshape(1, -1)
    
    # Compute cosine similarity with all tracks
    similarities = cosine_similarity(
        user_profile,
        df_features_scaled.values
    )[0]
    
    # Create results dataframe
    results = pd.DataFrame({
        'track_id': df_features_scaled.index,
        'similarity': similarities
    })
    
    # Filter out already seen tracks
    seen_ids = set(user_ratings.keys())
    results = results[~results['track_id'].isin(seen_ids)]
    
    # Filter out disliked artists and albums
    disliked_artists = set()
    disliked_albums = set()
    for tid in disliked_ids:
        disliked_artists.add(df_tracks.loc[tid, 'artist_name'])
        disliked_albums.add(df_tracks.loc[tid, 'album_name'])
    
    # Add metadata
    results['artist'] = results['track_id'].map(
        lambda x: df_tracks.loc[x, 'artist_name']
    )
    results['album'] = results['track_id'].map(
        lambda x: df_tracks.loc[x, 'album_name']
    )
    results['genre'] = results['track_id'].map(
        lambda x: df_tracks.loc[x, 'artist_primary_genre_broad']
    )
    results['popularity'] = results['track_id'].map(
        lambda x: df_tracks.loc[x, 'track_popularity']
    )
    
    # Filter: avoid disliked artists and albums
    results = results[
        (~results['artist'].isin(disliked_artists)) &
        (~results['album'].isin(disliked_albums))
    ]
    
    # Sort by similarity (descending)
    results = results.sort_values('similarity', ascending=False)
    
    # Greedy selection with diversity
    selected = []
    selected_artists = set()
    selected_genres = set()
    
    for _, row in results.iterrows():
        if len(selected) >= n_recommendations:
            break
        
        # Diversity penalty
        artist_penalty = 1.0
        genre_penalty = 1.0
        
        if diversity_weight > 0:
            if row['artist'] in selected_artists:
                artist_penalty = 1.0 - diversity_weight * 0.3
            if row['genre'] in selected_genres:
                genre_penalty = 1.0 - diversity_weight * 0.2
        
        # Adjusted similarity score
        adjusted_similarity = row['similarity'] * artist_penalty * genre_penalty
        
        selected.append({
            **row.to_dict(),
            'adjusted_similarity': adjusted_similarity
        })
        
        selected_artists.add(row['artist'])
        selected_genres.add(row['genre'])
    
    recommendations = pd.DataFrame(selected)
    recommendations = recommendations.sort_values('similarity', ascending=False)
    
    return recommendations

print("Advanced recommendation engine loaded with diversity constraints.")

Advanced recommendation engine loaded with diversity constraints.


In [14]:
# Generate recommendations using the advanced engine
print("Generating personalized recommendations...")
print(f"\nYour ratings:")
for track_id, rating in user_ratings.items():
    track_name = df_tracks.loc[track_id, 'track_name']
    artist = df_tracks.loc[track_id, 'artist_name']
    rating_str = 'LIKE' if rating == 1 else 'DISLIKE' if rating == -1 else 'SKIP'
    print(f"  {rating_str}: {track_name} - {artist}")

# Use advanced recommendation engine with diversity constraints
recommendations = get_recommendations_advanced(
    user_ratings,
    df_tracks,
    df_features_scaled,
    n_recommendations=10,
    diversity_weight=0.3
)

if len(recommendations) > 0:
    print(f"\nGenerated {len(recommendations)} diverse recommendations!")
    print(f"Unique artists: {recommendations['artist'].nunique()}")
    print(f"Unique genres: {recommendations['genre'].nunique()}")
else:
    print("\nCould not generate recommendations. Please like at least one track.")

Generating personalized recommendations...

Your ratings:
  DISLIKE: Pretty Hands - Jeffrey Foucault
  DISLIKE: I Understand I Do Not Understand Do You Understand? - Travelanguage
  LIKE: Big Jet Plane - Alok
  DISLIKE: Puller Opp - G.Flow
  LIKE: Outside - ËMMË
  DISLIKE: 2012 - Whoja Vu
  DISLIKE: Sugar - Rap Bang Club
  DISLIKE: Te Quiero un Poco - Carlos Sadness
  DISLIKE: Caught In The Rain - Revis
  LIKE: Old Friends - Juliana

Generated 10 diverse recommendations!
Unique artists: 9
Unique genres: 2


## 4A. Advanced Recommendation Engine with Diversity

## 5. Recommended Playlist

In [15]:
if len(recommendations) > 0:
    # Create playlist dataframe with full metadata
    playlist = []
    
    for idx, (_, rec) in enumerate(recommendations.iterrows(), 1):
        track_id = rec['track_id']
        track = df_tracks.loc[track_id]
        
        playlist.append({
            'Rank': idx,
            'Track': track['track_name'],
            'Artist': track['artist_name'],
            'Album': track['album_name'],
            'Genre': track['artist_primary_genre_broad'],
            'Popularity': track['track_popularity'],
            'Similarity': f"{rec['similarity']:.4f}"
        })
    
    playlist_df = pd.DataFrame(playlist)
    
    print("\n" + "=" * 100)
    print("YOUR PERSONALIZED PLAYLIST - TOP 10 RECOMMENDATIONS")
    print("=" * 100)
    print(playlist_df.to_string(index=False))
    print("=" * 100)
else:
    print("\nNo recommendations generated. Please complete the rating interface above.")


YOUR PERSONALIZED PLAYLIST - TOP 10 RECOMMENDATIONS
 Rank              Track             Artist           Album      Genre  Popularity Similarity
    1        Neon Colors      Arctic Vision   Arctic Vision                     48     0.9328
    2            Breathe Lonely in the Rain         Breathe                     54     0.9293
    3      Woman Go Wild       Pixie Geldof       I'm Yours                     48     0.9194
    4               Home       Gaspar Narby            Home                     39     0.9111
    5           U Adelie      Arctic Vision   Arctic Vision                     50     0.9011
    6    Fear of Heights         The Aprons Any Human Heart                      5     0.8910
    7       Slow Me Down              Mauve    Slow Me Down                     43     0.8829
    8            Dreamer              Lanam         Dreamer                     13     0.8817
    9 Je n'attendrai pas      Marie-Eve Roy     Multicolore                     25     0.8772
   10  

## 6. Playlist Visualization

In [16]:
if len(recommendations) > 0:
    # Similarity scores visualization
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=playlist_df['Track'],
        y=playlist_df['Similarity'].astype(float),
        marker=dict(
            color=playlist_df['Similarity'].astype(float),
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Similarity")
        ),
        text=[f"{artist}<br>{sim}" 
              for artist, sim in zip(playlist_df['Artist'], playlist_df['Similarity'])],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Artist: %{customdata}<br>Similarity: %{y:.4f}<extra></extra>',
        customdata=playlist_df['Artist']
    ))
    
    fig.update_layout(
        title="Recommendation Similarity Scores",
        xaxis_title="Track Name",
        yaxis_title="Cosine Similarity Score",
        height=500,
        showlegend=False,
        xaxis_tickangle=-45
    )
    
    fig.show()
    
    # Genre distribution
    genre_counts = playlist_df['Genre'].value_counts()
    
    fig_genre = go.Figure(data=[
        go.Pie(
            labels=genre_counts.index,
            values=genre_counts.values,
            hovertemplate='<b>%{label}</b><br>Count: %{value}<extra></extra>'
        )
    ])
    
    fig_genre.update_layout(
        title="Genre Distribution in Playlist",
        height=400
    )
    
    fig_genre.show()
    
    # Popularity distribution
    fig_pop = go.Figure()
    
    fig_pop.add_trace(go.Scatter(
        x=playlist_df['Rank'],
        y=playlist_df['Popularity'].astype(int),
        mode='markers+lines',
        marker=dict(
            size=10,
            color=playlist_df['Popularity'].astype(int),
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title="Popularity")
        ),
        line=dict(color='lightblue', width=2),
        hovertemplate='<b>%{customdata}</b><br>Rank: %{x}<br>Popularity: %{y}<extra></extra>',
        customdata=playlist_df['Track']
    ))
    
    fig_pop.update_layout(
        title="Track Popularity by Recommendation Rank",
        xaxis_title="Recommendation Rank",
        yaxis_title="Popularity Score",
        height=400
    )
    
    fig_pop.show()

In [17]:
if len(recommendations) > 0:
    # Compare audio features: Your likes vs Recommendations
    liked_ids = [tid for tid, rating in user_ratings.items() if rating == 1]
    
    # Calculate average audio features for your liked tracks
    liked_features = df_features_matrix.loc[liked_ids, audio_features].mean()
    
    # Calculate average audio features for recommendations
    rec_ids = recommendations['track_id'].tolist()
    rec_features = df_features_matrix.loc[rec_ids, audio_features].mean()
    
    # Normalize for visualization (0-1 scale per feature)
    feature_stats = df_features_matrix[audio_features].describe()
    liked_norm = (liked_features - feature_stats.loc['min']) / (feature_stats.loc['max'] - feature_stats.loc['min'])
    rec_norm = (rec_features - feature_stats.loc['min']) / (feature_stats.loc['max'] - feature_stats.loc['min'])
    
    # Radar chart
    import plotly.graph_objects as go
    
    fig_radar = go.Figure(data=[
        go.Scatterpolar(
            r=liked_norm.values,
            theta=audio_features,
            fill='toself',
            name='Your Preferences',
            line_color='#1f77b4'
        ),
        go.Scatterpolar(
            r=rec_norm.values,
            theta=audio_features,
            fill='toself',
            name='Recommendations',
            line_color='#ff7f0e'
        )
    ])
    
    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Audio Features Comparison: Your Likes vs Recommendations",
        showlegend=True,
        height=500
    )
    
    fig_radar.show()
    
    # Feature comparison table
    comparison = pd.DataFrame({
        'Feature': audio_features,
        'Your Preference': liked_features.values.round(3),
        'Recommendations': rec_features.values.round(3),
        'Difference': (rec_features - liked_features).values.round(3)
    })
    
    print("\nAudio Features Comparison:")
    print("=" * 80)
    print(comparison.to_string(index=False))
    print("=" * 80)


Audio Features Comparison:
         Feature  Your Preference  Recommendations  Difference
    acousticness            0.417            0.514       0.098
    danceability            0.579            0.577      -0.002
          energy            0.538            0.415      -0.123
instrumentalness            0.310            0.540       0.229
        liveness            0.122            0.108      -0.014
        loudness          -10.305          -12.067      -1.762
           tempo          114.375          114.167      -0.208
         valence            0.237            0.079      -0.158


## 6A. Audio Features Analysis

## 7. Recommendation Summary & Statistics

In [18]:
if len(recommendations) > 0:
    # Calculate similarity matrix between all recommended tracks
    rec_ids = recommendations['track_id'].tolist()
    rec_features_subset = df_features_scaled.loc[rec_ids]
    
    # Compute cosine similarity matrix
    sim_matrix = cosine_similarity(rec_features_subset)
    
    # Create heatmap
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=sim_matrix,
        x=[f"{i+1}" for i in range(len(rec_ids))],
        y=[f"{i+1}" for i in range(len(rec_ids))],
        colorscale='RdYlGn',
        zmid=0.5,
        hovertemplate='Track %{x} - Track %{y}: %{z:.3f}<extra></extra>'
    ))
    
    fig_heatmap.update_layout(
        title="Playlist Cohesion: Track-to-Track Similarity",
        xaxis_title="Recommendation #",
        yaxis_title="Recommendation #",
        height=500,
        width=600
    )
    
    fig_heatmap.show()
    
    # Cohesion statistics
    # Get upper triangle (excluding diagonal)
    triu_indices = np.triu_indices_from(sim_matrix, k=1)
    cohesion_scores = sim_matrix[triu_indices]
    
    print("\nPlaylist Cohesion Metrics:")
    print("=" * 80)
    print(f"Average inter-track similarity: {cohesion_scores.mean():.4f}")
    print(f"Min inter-track similarity:     {cohesion_scores.min():.4f}")
    print(f"Max inter-track similarity:     {cohesion_scores.max():.4f}")
    print(f"Std Dev:                        {cohesion_scores.std():.4f}")
    print("=" * 80)
    print("\nInterpretation:")
    if cohesion_scores.mean() > 0.7:
        print("  HIGH COHESION: Playlist is very consistent in style (good for focused listening)")
    elif cohesion_scores.mean() > 0.5:
        print("  MEDIUM COHESION: Playlist has variety with consistent themes")
    else:
        print("  HIGH DIVERSITY: Playlist is diverse (good for exploration)")


Playlist Cohesion Metrics:
Average inter-track similarity: 0.8998
Min inter-track similarity:     0.7556
Max inter-track similarity:     0.9795
Std Dev:                        0.0503

Interpretation:
  HIGH COHESION: Playlist is very consistent in style (good for focused listening)


## 6B. Playlist Cohesion Analysis

In [19]:
if len(recommendations) > 0:
    print("\n" + "=" * 80)
    print("PLAYLIST STATISTICS")
    print("=" * 80)
    
    # Calculate statistics
    similarity_scores = playlist_df['Similarity'].astype(float)
    popularity_scores = playlist_df['Popularity'].astype(int)
    
    print(f"\nSimilarity Metrics:")
    print(f"  Average Similarity:  {similarity_scores.mean():.4f}")
    print(f"  Min Similarity:      {similarity_scores.min():.4f}")
    print(f"  Max Similarity:      {similarity_scores.max():.4f}")
    print(f"  Std Dev:             {similarity_scores.std():.4f}")
    
    print(f"\nPopularity Metrics:")
    print(f"  Average Popularity:  {popularity_scores.mean():.1f}/100")
    print(f"  Min Popularity:      {popularity_scores.min()}/100")
    print(f"  Max Popularity:      {popularity_scores.max()}/100")
    
    print(f"\nDiversity:")
    print(f"  Unique Genres:       {playlist_df['Genre'].nunique()}")
    print(f"  Unique Artists:      {playlist_df['Artist'].nunique()}")
    print(f"  Unique Albums:       {playlist_df['Album'].nunique()}")
    
    # Liked tracks analysis
    liked_tracks = [tid for tid, rating in user_ratings.items() if rating == 1]
    if len(liked_tracks) > 0:
        liked_artists = df_tracks.loc[liked_tracks, 'artist_name'].unique()
        liked_genres = df_tracks.loc[liked_tracks, 'artist_primary_genre_broad'].unique()
        
        print(f"\nYour Preferences:")
        print(f"  Liked Artists:       {', '.join(liked_artists)}")
        print(f"  Liked Genres:        {', '.join(liked_genres)}")
    
    print("\n" + "=" * 80)


PLAYLIST STATISTICS

Similarity Metrics:
  Average Similarity:  0.9000
  Min Similarity:      0.8731
  Max Similarity:      0.9328
  Std Dev:             0.0221

Popularity Metrics:
  Average Popularity:  37.0/100
  Min Popularity:      5/100
  Max Popularity:      54/100

Diversity:
  Unique Genres:       2
  Unique Artists:      9
  Unique Albums:       9

Your Preferences:
  Liked Artists:       Alok, ËMMË, Juliana
  Liked Genres:        electronic, 



## 8. Export Playlist

In [ ]:
if len(recommendations) > 0:
    # Create export dataframe
    export_data = []
    
    for idx, (_, rec) in enumerate(recommendations.iterrows(), 1):
        track_id = rec['track_id']
        track = df_tracks.loc[track_id]
        
        export_data.append({
            'Rank': idx,
            'Track_ID': track_id,
            'Track_Name': track['track_name'],
            'Artist': track['artist_name'],
            'Album': track['album_name'],
            'Genre': track['artist_primary_genre_broad'],
            'Popularity': track['track_popularity'],
            'Similarity_Score': rec['similarity'],
            'Preview_URL': track['preview_url'],
            'Spotify_URI': track['uri']
        })
    
    export_df = pd.DataFrame(export_data)
    
    # Save to CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_file = project_root / 'data' / 'recommendations' / f'playlist_{timestamp}.csv'
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    export_df.to_csv(output_file, index=False)
    print(f"Playlist exported to: {output_file}")
    
    # Display preview
    print(f"\nExport preview (first 3 tracks):")
    print(export_df[['Rank', 'Track_Name', 'Artist', 'Similarity_Score']].head(3).to_string(index=False))